# Sales Prediction from Advertising Budgets

This notebook investigates whether product sales can be predicted from advertising budgets across TV, radio, and newspaper channels. It compares an interpretable linear baseline with a nonlinear Random Forest model, then evaluates both with MAE, RMSE, and R².

The analysis uses the supplied `Advertising Budget and Sales.csv` file. The unnamed first CSV column is an observation index and is intentionally excluded from modeling.

## 1. Imports and analysis setup

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (8, 5)

DATA_PATH = Path('Advertising Budget and Sales.csv')
RANDOM_STATE = 42

## 2. Load and inspect the data

In [ ]:
raw_data = pd.read_csv(DATA_PATH)

# Remove the unnamed CSV index column without relying on its exact generated name.
index_columns = [column for column in raw_data.columns if column.lower().startswith('unnamed')]
data = raw_data.drop(columns=index_columns).copy()

column_map = {
    'TV Ad Budget ($)': 'TV',
    'Radio Ad Budget ($)': 'Radio',
    'Newspaper Ad Budget ($)': 'Newspaper',
    'Sales ($)': 'Sales',
}
data = data.rename(columns=column_map)

features = ['TV', 'Radio', 'Newspaper']
target = 'Sales'

print(f'Shape: {data.shape}')
display(data.head())

In [ ]:
print('Missing values by column:')
display(data.isna().sum().to_frame('missing_values'))

print('Descriptive statistics:')
display(data.describe().T)

The missing-value check confirms whether any cleaning is needed before modeling. The descriptive statistics show the scale and spread of each advertising budget and of sales.

## 3. Exploratory data analysis

In [ ]:
sns.pairplot(data[features + [target]], corner=True, diag_kind='hist', plot_kws={'alpha': 0.7})
plt.suptitle('Pairplot of Advertising Budgets and Sales', y=1.02)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ['#1768AC', '#D1495B', '#00798C']
for axis, feature, color in zip(axes, features, colors):
    sns.scatterplot(data=data, x=feature, y=target, ax=axis, color=color, alpha=0.75)
    sns.regplot(data=data, x=feature, y=target, ax=axis, scatter=False, color='black', line_kws={'linewidth': 1.5})
    axis.set_title(f'{target} vs {feature} budget')
    axis.set_xlabel(f'{feature} advertising spend')
    axis.set_ylabel('Sales')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
correlation = data[features + [target]].corr()
sns.heatmap(correlation, annot=True, fmt='.2f', cmap='vlag', center=0, square=True)
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

The scatter plots and correlation matrix provide a first look at linear association. Correlation is useful for exploration, but model performance must be judged on data held out from training.

## 4. Train/test split

In [ ]:
X = data[features]
y = data[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

print(f'Training rows: {len(X_train)}')
print(f'Test rows: {len(X_test)}')

## 5. Model training and evaluation

The linear model is the baseline because its coefficients are easy to interpret. The Random Forest provides a second model that can represent nonlinear relationships and interactions without requiring manual feature transformations. Both models use the same train/test split for a fair comparison.

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(
        n_estimators=300, random_state=RANDOM_STATE, min_samples_leaf=2
    ),
}

results = []
predictions = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    predicted = model.predict(X_test)
    predictions[name] = predicted
    results.append({
        'Model': name,
        'MAE': mean_absolute_error(y_test, predicted),
        'RMSE': mean_squared_error(y_test, predicted) ** 0.5,
        'R2': r2_score(y_test, predicted),
    })

results_df = pd.DataFrame(results).sort_values('RMSE').reset_index(drop=True)
display(results_df.style.format({'MAE': '{:.3f}', 'RMSE': '{:.3f}', 'R2': '{:.3f}'}))

Lower MAE and RMSE indicate smaller prediction errors. Higher R² indicates that the model explains more of the variation in held-out sales.

## 6. Residual analysis for the best model

In [ ]:
best_model_name = results_df.loc[0, 'Model']
best_predictions = predictions[best_model_name]
residuals = y_test.to_numpy() - best_predictions

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(x=best_predictions, y=residuals, ax=axes[0], color='#3C6E71', s=65)
axes[0].axhline(0, color='#D1495B', linestyle='--')
axes[0].set_title(f'Residuals vs predictions: {best_model_name}')
axes[0].set_xlabel('Predicted sales')
axes[0].set_ylabel('Residual (actual - predicted)')

sns.histplot(residuals, kde=True, ax=axes[1], color='#1768AC')
axes[1].axvline(0, color='#D1495B', linestyle='--')
axes[1].set_title('Residual distribution')
axes[1].set_xlabel('Residual')
plt.tight_layout()
plt.show()

print(f'Best model by test RMSE: {best_model_name}')
print('A useful residual plot has points scattered around zero without a clear curve, funnel, or trend.')

Inspect the residual scatter for systematic structure. A curved pattern can suggest missing nonlinear effects, while a widening spread can suggest non-constant error variance. Since the test set is relatively small, this visual check should be treated as diagnostic rather than definitive.

## 7. Which channel has the highest impact?

In [ ]:
linear_model = models['Linear Regression']
coefficient_table = pd.DataFrame({
    'Channel': features,
    'Linear coefficient': linear_model.coef_,
    'Absolute coefficient': abs(linear_model.coef_),
}).sort_values('Absolute coefficient', ascending=False)

forest_model = models['Random Forest']
importance_table = pd.DataFrame({
    'Channel': features,
    'Random Forest importance': forest_model.feature_importances_,
}).sort_values('Random Forest importance', ascending=False)

print('Linear regression coefficients:')
display(coefficient_table.style.format({'Linear coefficient': '{:.4f}', 'Absolute coefficient': '{:.4f}'}))
print('Random Forest feature importances:')
display(importance_table.style.format({'Random Forest importance': '{:.4f}'}))

linear_leader = coefficient_table.iloc[0]['Channel']
forest_leader = importance_table.iloc[0]['Channel']
print(f'Largest absolute linear coefficient: {linear_leader}')
print(f'Largest Random Forest importance: {forest_leader}')
print('Interpretation: the channel ranked highest by the chosen measure is the strongest signal in this dataset; this is association, not proof that increasing spend alone causes the increase.')

## 8. Conclusion

The evaluation table identifies which model generalizes better on the held-out data. The coefficient and feature-importance tables answer the channel-impact question from two complementary perspectives: linear marginal association and tree-based predictive usefulness. Results should be revisited with cross-validation and newer campaign data before making budget decisions.